In [32]:
using Statistics, DataFrames, Distributions

data = [
    [3939.237, 3710.146, 3961.150],
    [20020.892, 19430.791, 32769.866],
    [4117.070, 4128.740, 4274.476],
    [21173.682, 21301.682, 22729.838],
    [4873.346, 4918.758, 4995.244],
    [27273.054, 27575.705, 27910.514],
    [4918.758, 4941.586, 4964.414],
    [31077.958, 31970.035, 31191.336],
    [4738.029, 4747.114, 4756.200],
    [34690.074, 35191.039, 34912.278],
    [4557.382, 4584.938, 4612.495],
    [37773.254, 38340.109, 37990.559]
]

# === FUNCTIONS ===
function grubbs_test(x; α=0.10)
    n = length(x)
    x̄, s = mean(x), std(x)
    G = maximum(abs.(x .- x̄)) / s
    tcrit = quantile(TDist(n - 2), 1 - α / (2n))
    Gcrit = ((n - 1) / sqrt(n)) * sqrt(tcrit^2 / (n - 2 + tcrit^2))
    return (argmax(abs.(x .- x̄)), G > Gcrit, G, Gcrit)
end

function dixons_q_test(x; α=0.10)
    n = length(x)
    sorted = sort(x)
    Qcrit = Dict(3=>0.941, 4=>0.765, 5=>0.642)[n]
    Q_high = (sorted[end] - sorted[end-1]) / (sorted[end] - sorted[1])
    Q_low  = (sorted[2] - sorted[1]) / (sorted[end] - sorted[1])
    if Q_high > Q_low
        return (findfirst(==(sorted[end]), x), Q_high > Qcrit, Q_high, Qcrit)
    else
        return (findfirst(==(sorted[1]), x), Q_low > Qcrit, Q_low, Qcrit)
    end
end

function heuristic_outlier(x; thresh=2.0)
    x̄, s = mean(x), std(x)
    diffs = abs.(x .- x̄)
    idx = findfirst(>(thresh*s), diffs)
    return isnothing(idx) ? (0, false) : (idx, true)
end

# === APPLY ALL THREE ===
results = DataFrame(Sample=Int[], GrubbsIndex=Int[], GrubbsFlag=Bool[],
                    DixonIndex=Int[], DixonFlag=Bool[],
                    HeuristicIndex=Int[], HeuristicFlag=Bool[],
                    CombinedFlag=Bool[])

for (i, row) in enumerate(data)
    gi, gflag, G, Gcrit = grubbs_test(row)
    di, dflag, Q, Qcrit = dixons_q_test(row)
    hi, hflag = heuristic_outlier(row)
    combined = gflag || dflag || hflag
    push!(results, (i, gi, gflag, di, dflag, hi, hflag, combined))
end

println("=== Outlier Detection Summary ===")
println(results)

println("\nFlagged samples:")
for i in 1:size(results,1)
    if results.CombinedFlag[i]
        println("Sample $(results.Sample[i]): values = $(round.(data[i], digits=3))")
    end
end

=== Outlier Detection Summary ===
12×8 DataFrame
 Row │ Sample  GrubbsIndex  GrubbsFlag  DixonIndex  DixonFlag  HeuristicIndex  HeuristicFlag  CombinedFlag 
     │ Int64   Int64        Bool        Int64       Bool       Int64           Bool           Bool         
─────┼─────────────────────────────────────────────────────────────────────────────────────────────────────
   1 │      1            2       false           2      false               0          false         false
   2 │      2            3        true           3       true               0          false          true
   3 │      3            3       false           3      false               0          false         false
   4 │      4            3       false           3      false               0          false         false
   5 │      5            3       false           3      false               0          false         false
   6 │      6            3       false           3      false               0          false